# 02 — One-arm profile for the laptop

This notebook profiles the cleaned one-arm implementation on a medium-sized
problem using the physical LISA orbit file.

It measures sequential versus parallel WDM transforms, ordinary materialized
six-link shifting, parallel-link schedules, and the complete A1 stage.

Results are saved directly beside the notebook with a `_local_wdm_` prefix.

In [1]:

import os
import platform

import jax.numpy as jnp

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

import json
import platform
import subprocess
import time
from pathlib import Path

import jax
import numpy as np

from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

import WDM_LISAresponse


NOTEBOOK_DIR = Path.cwd().resolve()
RESPONSE_ROOT = next(
    (
        path for path in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
        if (path / "examples").exists()
        and ((path / "src" / "WDM_LISAresponse").exists()
             or (path / "WDM_LISAresponse").exists())
    ),
    None,
)
if RESPONSE_ROOT is None:
    # Installed-package fallback.
    RESPONSE_ROOT = Path(WDM_LISAresponse.__file__).resolve().parents[2]

ORBIT_FILE = RESPONSE_ROOT / "examples" / "esa-trailing-orbits.h5"
if not ORBIT_FILE.exists():
    raise FileNotFoundError(
        "Could not find the orbit file at "
        f"{ORBIT_FILE}. The file should remain in WDM_LISAresponse/examples."
    )


def load_lisa_orbits(path: Path):
    """Load the physical LISA orbit file used by the response calculations."""
    orbit_data = read_orbit_file(str(path))
    orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
    orbits = InterpolatedOrbits(
        orbit_times,
        orbit_data.positions,
        orbit_data.velocities,
    )
    return orbit_data, orbits


def git_state(package_file: str) -> dict[str, object]:
    """Return the current Git commit and whether the checkout is dirty."""
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(Path(package_file).resolve().parent),
            "rev-parse",
            "--show-toplevel",
        ],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        return {"root": None, "commit": None, "dirty": None}

    root = Path(completed.stdout.strip()).resolve()
    commit = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    dirty = bool(
        subprocess.run(
            ["git", "-C", str(root), "status", "--porcelain"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    )
    return {"root": str(root), "commit": commit, "dirty": dirty}


def json_ready(value):
    """Convert NumPy and JAX objects into JSON-compatible values."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item") and getattr(value, "ndim", None) == 0:
        return value.item()
    if hasattr(value, "tolist"):
        return value.tolist()
    return value


def save_json(filename: str, payload) -> Path:
    """Save a JSON result beside the notebook."""
    path = NOTEBOOK_DIR / filename
    path.write_text(
        json.dumps(json_ready(payload), indent=2),
        encoding="utf-8",
    )
    print("Saved:", path)
    return path


orbit_data, orbits = load_lisa_orbits(ORBIT_FILE)
YEAR_SI = 365.25 * 24.0 * 3600.0
t0 = float(orbit_data.t0 + YEAR_SI)

print("Notebook directory:", NOTEBOOK_DIR)
print("Orbit file:", ORBIT_FILE)
print("Python:", platform.python_version())
print("JAX:", jax.__version__, "| backend:", jax.default_backend())
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("Source state:", git_state(WDM_LISAresponse.__file__))


from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMStaticPlan
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_context_from_wdm,
    _build_wdm_dT_grid,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)

def block_result(value):
    """Synchronise every JAX leaf before stopping a timer."""
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def timed(function):
    started = time.perf_counter()
    value = function()
    block_result(value)
    return value, float(time.perf_counter() - started)


def timing_summary(values):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median_s": float(np.median(values)),
        "q25_s": float(q25),
        "q75_s": float(q75),
        "iqr_s": float(q75 - q25),
        "minimum_s": float(np.min(values)),
        "maximum_s": float(np.max(values)),
        "all_s": [float(value) for value in values],
    }


def benchmark(function, *, warmups: int, repeats: int):
    for _ in range(int(warmups)):
        block_result(function())

    samples = []
    latest = None
    for _ in range(int(repeats)):
        latest, elapsed = timed(function)
        samples.append(elapsed)
    return latest, timing_summary(samples)


def compare_arrays(candidate, reference, *, m_start: int = 0):
    """Return max-absolute and relative-L2 differences."""
    candidate = jax.numpy.asarray(candidate)[..., m_start:]
    reference = jax.numpy.asarray(reference)[..., m_start:]
    difference = candidate - reference

    difference_power = jax.numpy.sum(jax.numpy.abs(difference) ** 2)
    reference_power = jax.numpy.sum(jax.numpy.abs(reference) ** 2)
    max_abs = jax.numpy.max(jax.numpy.abs(difference))
    rel_l2 = jax.numpy.where(
        reference_power > 0,
        jax.numpy.sqrt(difference_power / reference_power),
        jax.numpy.sqrt(difference_power),
    )
    block_result((max_abs, rel_l2))
    return {"max_abs": float(max_abs), "rel_l2": float(rel_l2)}


def compare_link_maps(candidate, reference, *, m_start: int = 0):
    if set(candidate) != set(reference):
        raise ValueError("Directed-link dictionaries contain different keys.")

    metrics = {
        str(link): compare_arrays(
            candidate[link],
            reference[link],
            m_start=m_start,
        )
        for link in reference
    }
    return {
        "per_link": metrics,
        "max_abs": max(item["max_abs"] for item in metrics.values()),
        "rel_l2": max(item["rel_l2"] for item in metrics.values()),
    }


process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    return (
        process.memory_info().rss / 1024**3
        if process is not None
        else float("nan")
    )


print("Machine:", platform.node())
print("CPU:", platform.processor())
print(
    "Physical cores:",
    psutil.cpu_count(logical=False) if psutil is not None else "unknown",
)


Notebook directory: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples
Orbit file: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples/esa-trailing-orbits.h5
Python: 3.13.11
JAX: 0.7.0 | backend: cpu
WDM_LISAresponse: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
Source state: {'root': '/Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse', 'commit': '155979a0a509bc82690c867631655d2434783661', 'dirty': True}
Machine: Hannas-MacBook-Pro-586.local
CPU: i386
Physical cores: 8


In [2]:
# Medium portable configuration

N = 320_000
dt = 10.0
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25
CALC_M0 = False

theta = 0.9
phi = 0.4

SHIFT_KWARGS = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": 3_200,
    "lag_block_size": 17,
    "batch_chunk": 6,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

DWT_WARMUPS = 1
DWT_REPEATS = 5
SHIFT_WARMUPS = 1
SHIFT_REPEATS = 5
PIPELINE_WARMUPS = 1
PIPELINE_REPEATS = 5

# Keep the laptop sweep small. Add a schedule only when the machine has
# enough physical cores to make it meaningful.
physical_cores = (
    psutil.cpu_count(logical=False)
    if psutil is not None
    else 4
)
PARALLEL_SCHEDULES = [
    {"name": "groups_3_workers_2", "group_size": 3, "workers": 2},
]
if physical_cores >= 6:
    PARALLEL_SCHEDULES.append(
        {"name": "groups_2_workers_3", "group_size": 2, "workers": 3}
    )

SAVE_RESULTS = True


In [3]:
# Generate one fixed waveform and build the reusable static one-arm plan.

times_pad, h_plus, h_cross, padding = generate_windowed_padded_waveform(
    waveform_model="monochromatic",
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs={
        "A_plus": 1.0e-21,
        "A_cross": 0.8e-21,
        "f": 2.0e-3,
        "phi_0": 0.2,
    },
    window_function="tukey",
    window_kwargs={"alpha": 0.01, "tukey_factor": 2.0},
    wdm_Nf=Nf,
    return_padding_info=True,
)

grid_context, grid_s = timed(
    lambda: _build_wdm_dT_grid(
        times_pad,
        Nf=Nf,
        d=d_wdm,
        A_frac=A_frac,
        q=q_wdm,
        calc_m0=CALC_M0,
    )
)
one_arm_plan, plan_s = timed(
    lambda: OneArmWDMStaticPlan.build_from_context(
        grid_context,
        orbits,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=SHIFT_KWARGS,
    )
)

print("WDM shape:", (grid_context["wdm"].Nt, grid_context["wdm"].Nf))
print("Grid build_s:", f"{grid_s:.3f}")
print("Static plan build_s:", f"{plan_s:.3f}")
print(
    "Interpolation-table memory [MiB]:",
    one_arm_plan.kernel_memory_bytes / 1024**2,
)
print("RSS [GiB]:", f"{rss_gib():.2f}")


WDM shape: (3264, 100)
Grid build_s: 1.700
Plan build_s: 0.459
Plan memory [MiB]: 15.315135955810547
RSS [GiB]: 0.56


## 1. Sequential versus parallel WDM transforms


In [4]:
def sequential_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=False,
    )


def parallel_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=True,
    )


sequential_context, sequential_dwt_stats = benchmark(
    sequential_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)
parallel_context, parallel_dwt_stats = benchmark(
    parallel_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)

dwt_plus = compare_arrays(
    parallel_context["w_t_plus"],
    sequential_context["w_t_plus"],
    m_start=1,
)
dwt_cross = compare_arrays(
    parallel_context["w_t_cross"],
    sequential_context["w_t_cross"],
    m_start=1,
)

print("Sequential DWT:", sequential_dwt_stats)
print("Parallel DWT:", parallel_dwt_stats)
print("Plus comparison:", dwt_plus)
print("Cross comparison:", dwt_cross)


Sequential DWT: {'median_s': 0.017597458005184308, 'q25_s': 0.0175253750057891, 'q75_s': 0.019073582981945947, 'iqr_s': 0.0015482079761568457, 'minimum_s': 0.017322791973128915, 'maximum_s': 0.01958245798596181, 'all_s': [0.01958245798596181, 0.019073582981945947, 0.017597458005184308, 0.0175253750057891, 0.017322791973128915]}
Parallel DWT: {'median_s': 0.011738041997887194, 'q25_s': 0.011503334011649713, 'q75_s': 0.011997875000815839, 'iqr_s': 0.0004945409891661257, 'minimum_s': 0.011285832995781675, 'maximum_s': 0.012328459008131176, 'all_s': [0.011503334011649713, 0.011285832995781675, 0.011997875000815839, 0.011738041997887194, 0.012328459008131176]}
Plus comparison: {'max_abs': 0.0, 'rel_l2': 0.0}
Cross comparison: {'max_abs': 0.0, 'rel_l2': 0.0}


## 2. One-arm link scheduling


In [5]:
w_plus = jnp.asarray(parallel_context["w_t_plus"])
w_cross = jnp.asarray(parallel_context["w_t_cross"])
block_result((w_plus, w_cross))

def runtime_sky_shift():
    return one_arm_plan.apply_device(
        w_plus,
        w_cross,
        theta,
        phi,
    )

baseline_links, baseline_shift_stats = benchmark(
    runtime_sky_shift,
    warmups=SHIFT_WARMUPS,
    repeats=SHIFT_REPEATS,
)

schedule_rows = [
    {
        "name": "runtime_sky",
        **baseline_shift_stats,
    }
]
best_schedule = schedule_rows[0]

print("Runtime sky + one-arm:", baseline_shift_stats)


,name,group_size,workers,median_s,q25_s,q75_s,iqr_s,minimum_s,maximum_s,all_s,max_abs,rel_l2
0,materialized,6,1,0.182610,0.181227,0.183145,0.001917,0.163842,0.216445,"[0.16384158298023976, 0.18122733299969696, 0.2...",0.0,0.0
1,groups_3_workers_2,3,2,0.113722,0.109608,0.115847,0.006240,0.107419,0.204751,"[0.10960766699281521, 0.11372200000914745, 0.1...",0.0,0.0
2,groups_2_workers_3,2,3,0.104582,0.101761,0.131203,0.029441,0.089766,0.140415,"[0.10458174999803305, 0.10176108402083628, 0.1...",0.0,0.0


Selected laptop schedule: groups_2_workers_3


## 3. Complete A1 stage


In [6]:
def baseline_a1():
    context = sequential_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        theta,
        phi,
    )

def selected_a1():
    context = parallel_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        theta,
        phi,
    )

baseline_a1_output, baseline_a1_stats = benchmark(
    baseline_a1,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)
selected_a1_output, selected_a1_stats = benchmark(
    selected_a1,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)

a1_comparison = compare_link_maps(
    selected_a1_output,
    baseline_a1_output,
    m_start=1,
)

print("Sequential-DWT runtime:", baseline_a1_stats)
print("Parallel-DWT runtime:", selected_a1_stats)
print("A1 comparison:", a1_comparison)


Baseline A1: {'median_s': 0.17879320800420828, 'q25_s': 0.17347954100114293, 'q75_s': 0.1818805840157438, 'iqr_s': 0.008401043014600873, 'minimum_s': 0.17147616698639467, 'maximum_s': 0.18572224999661557, 'all_s': [0.1818805840157438, 0.18572224999661557, 0.17879320800420828, 0.17147616698639467, 0.17347954100114293]}
Selected A1: {'median_s': 0.10349054099060595, 'q25_s': 0.10328379200655036, 'q75_s': 0.1055737919814419, 'iqr_s': 0.0022899999748915434, 'minimum_s': 0.09992179099936038, 'maximum_s': 0.11917604098562151, 'all_s': [0.10328379200655036, 0.09992179099936038, 0.10349054099060595, 0.1055737919814419, 0.11917604098562151]}
A1 comparison: {'per_link': {'(2, 1)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 2)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(1, 3)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 1)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(2, 3)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(1, 2)': {'max_abs': 0.0, 'rel_l2': 0.0}}, 'max_abs': 0.0, 'rel_l2': 0.0}


In [7]:
results = {
    "source": git_state(Path(WDM_LISAresponse.__file__)),
    "machine": {
        "node": platform.node(),
        "processor": platform.processor(),
        "physical_cores": physical_cores,
    },
    "configuration": {
        "N": N,
        "dt": dt,
        "Nf": Nf,
        "L_trunc": L_TRUNC,
        "calc_m0": CALC_M0,
        "shift_kwargs": SHIFT_KWARGS,
    },
    "setup_s": {"grid": grid_s, "one_arm_static_plan": plan_s},
    "dwt": {
        "sequential": sequential_dwt_stats,
        "parallel": parallel_dwt_stats,
        "plus_comparison": dwt_plus,
        "cross_comparison": dwt_cross,
    },
    "shift_schedules": schedule_rows,
    "selected_schedule": best_schedule,
    "complete_a1": {
        "baseline": baseline_a1_stats,
        "selected": selected_a1_stats,
        "comparison": a1_comparison,
    },
    "rss_gib": rss_gib(),
}

print(json.dumps(json_ready(results), indent=2))


if SAVE_RESULTS:
    save_json("_local_wdm_02_laptop_profile.json", results)
    if pd is not None:
        schedule_path = (
            NOTEBOOK_DIR / "_local_wdm_02_laptop_shift_schedule.csv"
        )
        pd.DataFrame(schedule_rows).to_csv(
            schedule_path,
            index=False,
        )
        print("Saved:", schedule_path)


{
  "source": {
    "root": "/Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse",
    "commit": "155979a0a509bc82690c867631655d2434783661",
    "dirty": true
  },
  "machine": {
    "node": "Hannas-MacBook-Pro-586.local",
    "processor": "i386",
    "physical_cores": 8
  },
  "configuration": {
    "N": 320000,
    "dt": 10.0,
    "Nf": 100,
    "L_trunc": 25,
    "calc_m0": false,
    "shift_kwargs": {
      "assembly_backend": "production",
      "assembly_precision": "complex64",
      "row_chunk_size": 3200,
      "lag_block_size": 17,
      "batch_chunk": 6,
      "tl_tp_mode": "interp",
      "tl_tp_interp_points": 32,
      "tl_tp_interp_kind": "linear"
    }
  },
  "setup_s": {
    "grid": 1.7001638329820707,
    "one_arm_plan": 0.45939612499205396
  },
  "dwt": {
    "sequential": {
      "median_s": 0.017597458005184308,
      "q25_s": 0.0175253750057891,
      "q75_s": 0.019073582981945947,
      "iqr_s": 0.0015482079761568457,
      "minimum_s": 0.017322791973128915